# 03d · Validation 선택과 배포 봉인

이 파일은 DCC 2026 작업 노트북의 **공개용 보존본**입니다. 아래 원래 셀과 텍스트 출력은 저장된 실험 기록이며, 이번 공개 작업에서 모델 학습·추론·평가를 다시 실행하지 않았습니다.

- **목적:** validation RMSE로 후보 또는 유형별 router를 선택하고 full 승격의 상위 선택·학습 범위를 검증합니다.
- **입력:** 03B/03C candidate bundle, dataset seal; full_promote에서는 pilot selection과 full 학습 산출물
- **출력:** validation leaderboard와 선택 seal
- **실제 저장된 실행 상태:** 현재 설정은 full_promote이며 저장 승자는 type-router-convnext-tiny-yolo26s-roi-v1, validation RMSE=0.21439652743518847 m입니다. pilot 비교와 full 재학습을 같은 조건의 한 번 실행으로 해석하지 마십시오. 필요한 중간 pilot/full 산출물은 이 저장소에 모두 포함되어 있지 않습니다.

**실행 전:** 공개 저장소에는 원천 이미지·라벨 ZIP, private test 연결표·정답, 모델 가중치, 원래 GCS 산출물이 없습니다. `YOUR_GCP_PROJECT_ID`, `YOUR_GCS_BUCKET`과 필요한 데이터·권한을 직접 설정해야 하며, 원래 봉인 파일의 generation·SHA-256·config 검증도 충족해야 합니다. placeholder만 바꾼다고 과거 산출물이 재생성되거나 모든 gate가 통과하는 것은 아닙니다.

**공개본 변경:** 개인 프로젝트·버킷·계정·로컬 데이터 경로를 placeholder로 바꾸고 Colab/Drive/작성자/widget metadata를 제거했습니다. 코드 셀의 순서와 계산·무결성 검증 로직은 보존했습니다. 모든 내장 image/HTML/JavaScript/기타 binary 출력과 markdown 첨부 이미지를 제거하고, 숫자·표·오류를 포함한 텍스트 출력은 식별자를 가린 뒤 보존했습니다. 사진이 없는 것은 해당 셀을 실행하지 않았다는 뜻이 아닙니다. 원본 code cell의 `execution_count`는 모두 `null`이므로 이 값만으로 실행 여부를 판단하지 않습니다.

[GitHub 코드](https://github.com/Pirite08/dcc-2026-water-level-estimation/blob/main/notebooks/03d_model_selection.ipynb) · [Colab에서 열기](https://colab.research.google.com/github/Pirite08/dcc-2026-water-level-estimation/blob/main/notebooks/03d_model_selection.ipynb) · [실행 관계와 상태](README.md) · [원본 이름·SHA-256 대응표](../docs/notebook-source-map.csv)

---

아래는 비공개 식별자를 치환한 원래 노트북입니다. 원래 안내의 예정 단계·성공 기준은 위의 실제 저장 상태와 구분해 읽으십시오.


# DCC 2026 Mission 3 — 03D. Validation 후보 비교와 모델 선택 봉인

이 노트북은 test object를 읽지 않고 validation prediction만 재검증합니다. 전체 meter RMSE,
최악 유형 RMSE 순으로 승자를 정하고 checkpoint·전처리·fallback 규칙을 create-only seal에
고정합니다.

03B의 세 full-frame 후보와 03C의 YOLO26s+ROI 후보를 같은 표에서 비교합니다. 또한
validation 유형별 RMSE가 더 낮은 component를 고정 선택하는 `type_router` 후보를 합성합니다.
라우팅은 pilot에서만 정하고 full 재학습과 익명 test에서는 그대로 보존합니다.


## 런타임

표준 CPU 런타임을 사용합니다. 이 노트북에서 학습이나 test 추론을 수행하지 않습니다.


In [ ]:
%pip -q install "google-cloud-storage>=2.19,<4" "pyarrow>=18,<24" "pandas>=2.2,<3"


In [ ]:
import hashlib
import json
import math
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from google.colab import auth

PROJECT_ID = "YOUR_GCP_PROJECT_ID"
BUCKET_NAME = "YOUR_GCS_BUCKET"
SELECTION_MODE = "full_promote"  # "pilot_select" 또는 "full_promote"
PILOT_SELECTION_ID = "m3-selection-pilot-router-v1"
PILOT_INCLUDE_YOLO_CANDIDATE = True  # type-router 구성에 필요하므로 True를 유지합니다.
ROUTER_MODEL_ID = "type-router-convnext-tiny-yolo26s-roi-v1"
ROUTER_COMPONENT_IDS = {
    "full_frame": "fullframe-convnext_tiny",
    "yolo_roi": "yolo26s-roi-convnext-tiny",
}
ROUTING_KEY = "filename_first_character"
if SELECTION_MODE == "full_promote":
    DATASET_VERSION = "m3ds-v2-pilot"
    BASELINE_RUN_ID = "m3-fullframe-router-full-v2"
    YOLO_RUN_ID = "m3-yolo26s-roi-router-full-v2"
    INCLUDE_YOLO_CANDIDATE = False  # pilot seal의 winner kind에 따라 아래에서 결정
    SELECTION_ID = "m3-selection-full-router-v2"
else:
    DATASET_VERSION = "m3ds-v1-pilot"
    BASELINE_RUN_ID = "m3-fullframe-pilot-v1"
    YOLO_RUN_ID = "m3-yolo26s-roi-pilot-v1"
    INCLUDE_YOLO_CANDIDATE = PILOT_INCLUDE_YOLO_CANDIDATE
    SELECTION_ID = PILOT_SELECTION_ID
EXPECTED_VAL_N = 30_808
TYPE_ORDER = list("ABCEFGHS")

LOCAL_ROOT = Path("/content/dcc_mission3_03d")
LOCAL_META = LOCAL_ROOT / "meta"
LOCAL_INPUT = LOCAL_ROOT / "input"
for path in [LOCAL_META, LOCAL_INPUT]:
    path.mkdir(parents=True, exist_ok=True)
SELECTION_PREFIX = f"dcc/mission3/selection/{SELECTION_ID}"


In [ ]:
from google.api_core.exceptions import NotFound, PreconditionFailed
from google.cloud import storage

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        while True:
            chunk = stream.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

def canonical_sha256(value):
    raw = json.dumps(
        value, ensure_ascii=False, sort_keys=True, separators=(",", ":"),
        allow_nan=False,
    ).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()

def canonical_target_sha256(frame, target_column):
    """원본 float64와 DataLoader float32가 같은 수위를 같은 byte 계약으로 해시합니다."""
    required = {"stem", "type", target_column}
    missing = required - set(frame.columns)
    if missing:
        raise AssertionError(f"target hash 필수 열 누락: {sorted(missing)}")
    ordered = frame.sort_values("stem", kind="mergesort")
    records = [
        {
            "stem": str(getattr(row, "stem")),
            "type": str(getattr(row, "type")),
            "target_m_float32": float(np.float32(float(getattr(row, target_column)))),
        }
        for row in ordered.itertuples(index=False)
    ]
    return canonical_sha256(records)

def parse_gs_uri(uri):
    if not str(uri).startswith("gs://"):
        raise ValueError(f"GCS URI가 아닙니다: {uri}")
    bucket_name, object_name = str(uri)[5:].split("/", 1)
    return bucket_name, object_name

def download_object(bucket, object_name, local_path, generation=None):
    local_path = Path(local_path)
    local_path.parent.mkdir(parents=True, exist_ok=True)
    blob = bucket.blob(object_name, generation=generation)
    kwargs = {"checksum": "auto"}
    if generation is not None:
        kwargs["if_generation_match"] = int(generation)
    blob.download_to_filename(str(local_path), **kwargs)
    return local_path

def upload_create_only(bucket, local_path, object_name, artifact_type):
    local_path = Path(local_path)
    file_sha = sha256_file(local_path)
    blob = bucket.blob(object_name)
    blob.metadata = {
        "sha256": file_sha,
        "artifact_type": artifact_type,
    }
    try:
        blob.upload_from_filename(
            str(local_path), if_generation_match=0, checksum="auto"
        )
    except PreconditionFailed:
        blob.reload()
        remote_sha = (blob.metadata or {}).get("sha256")
        remote_artifact_type = (blob.metadata or {}).get("artifact_type")
        if (
            remote_sha != file_sha
            or remote_artifact_type != artifact_type
            or int(blob.size or -1) != local_path.stat().st_size
        ):
            raise RuntimeError(
                f"create-only 충돌: gs://{bucket.name}/{object_name}"
            )
    blob.reload()
    if (
        (blob.metadata or {}).get("sha256") != file_sha
        or (blob.metadata or {}).get("artifact_type") != artifact_type
    ):
        raise RuntimeError(f"업로드 SHA metadata 검증 실패: {object_name}")
    return {
        "uri": f"gs://{bucket.name}/{object_name}",
        "generation": int(blob.generation),
        "bytes": int(blob.size),
        "sha256": file_sha,
        "artifact_type": artifact_type,
    }

def upload_json_create_only(bucket, payload, object_name, artifact_type):
    local_path = LOCAL_META / Path(object_name).name
    local_path.parent.mkdir(parents=True, exist_ok=True)
    local_path.write_text(
        json.dumps(
            payload, ensure_ascii=False, sort_keys=True, indent=2, allow_nan=False
        ) + "\n",
        encoding="utf-8",
    )
    return upload_create_only(bucket, local_path, object_name, artifact_type)

def download_json(bucket, object_name, generation=None):
    blob = bucket.blob(object_name, generation=generation)
    kwargs = {}
    if generation is not None:
        kwargs["if_generation_match"] = int(generation)
    return json.loads(blob.download_as_bytes(**kwargs).decode("utf-8"))

def remote_descriptor(bucket, object_name, artifact_type=None):
    blob = bucket.blob(object_name)
    blob.reload()
    metadata = blob.metadata or {}
    file_sha = metadata.get("sha256")
    if not file_sha:
        raise RuntimeError(f"remote object에 sha256 metadata가 없습니다: {object_name}")
    remote_artifact_type = metadata.get("artifact_type")
    if artifact_type is not None and remote_artifact_type != artifact_type:
        raise RuntimeError(
            f"remote artifact_type 불일치: {object_name} "
            f"({remote_artifact_type!r} != {artifact_type!r})"
        )
    return {
        "uri": f"gs://{bucket.name}/{blob.name}",
        "generation": int(blob.generation),
        "bytes": int(blob.size),
        "sha256": file_sha,
        "artifact_type": remote_artifact_type or "unknown",
    }


In [ ]:
auth.authenticate_user()
STORAGE_CLIENT = storage.Client(project=PROJECT_ID)
BUCKET = STORAGE_CLIENT.bucket(BUCKET_NAME)
BUCKET.reload()

dataset_seal = download_json(
    BUCKET, f"dcc/mission3/datasets/{DATASET_VERSION}/dataset_seal.json"
)
parent_selection = None
parent_selection_descriptor = None
if SELECTION_MODE == "full_promote":
    parent_selection_object = (
        f"dcc/mission3/selection/{PILOT_SELECTION_ID}/model_selection_seal.json"
    )
    parent_selection_descriptor = remote_descriptor(
        BUCKET, parent_selection_object, "m3_model_selection_seal"
    )
    parent_selection_path = LOCAL_INPUT / "pilot_model_selection_seal.json"
    download_object(
        BUCKET,
        parent_selection_object,
        parent_selection_path,
        generation=parent_selection_descriptor["generation"],
    )
    if parent_selection_path.stat().st_size != int(parent_selection_descriptor["bytes"]):
        raise AssertionError("pilot selection seal byte size 불일치")
    if sha256_file(parent_selection_path) != parent_selection_descriptor["sha256"]:
        raise AssertionError("pilot selection seal SHA 불일치")
    parent_selection = json.loads(
        parent_selection_path.read_text(encoding="utf-8")
    )
    if parent_selection["status"] != "MODEL_SELECTION_FROZEN":
        raise AssertionError("pilot selection seal 상태 오류")
    if parent_selection.get("schema_version") != "dcc-m3-model-selection-seal-v3":
        raise AssertionError("pilot selection seal schema 오류")
    if parent_selection.get("selection_id") != PILOT_SELECTION_ID:
        raise AssertionError("pilot selection ID 오류")
    if parent_selection.get("dataset_version") != "m3ds-v1-pilot":
        raise AssertionError("pilot selection dataset version 오류")
    if parent_selection.get("selection_mode") != "pilot_select":
        raise AssertionError("pilot selection mode 오류")
    if parent_selection.get("training_scope") != "pilot":
        raise AssertionError("pilot selection training scope 오류")
    parent_assertions = parent_selection.get("assertions", {})
    if not parent_assertions or not all(parent_assertions.values()):
        raise AssertionError("pilot selection assertion 실패")
    if parent_selection.get("winner_model_id") != parent_selection["winner"].get("model_id"):
        raise AssertionError("pilot selection winner ID 오류")
    pilot_architecture_payload = parent_selection["winner"].get("architecture_payload")
    if (
        pilot_architecture_payload is not None
        and canonical_sha256(pilot_architecture_payload)
        != parent_selection["winner"].get("architecture_fingerprint")
    ):
        raise AssertionError("pilot winner architecture fingerprint 오류")
    frozen_kind = parent_selection["winner"]["kind"]
    if frozen_kind == "type_router":
        bundle_objects = [
            f"dcc/mission3/runs/{BASELINE_RUN_ID}/candidate_bundle.json",
            f"dcc/mission3/runs/{YOLO_RUN_ID}/candidate_bundle.json",
        ]
    elif frozen_kind == "yolo_roi":
        bundle_objects = [f"dcc/mission3/runs/{YOLO_RUN_ID}/candidate_bundle.json"]
    elif frozen_kind == "full_frame":
        bundle_objects = [f"dcc/mission3/runs/{BASELINE_RUN_ID}/candidate_bundle.json"]
    else:
        raise RuntimeError(f"지원하지 않는 pilot winner kind: {frozen_kind}")
    if dataset_seal["data_profile"] != "full" or dataset_seal["counts"]["train"] < 200_000:
        raise AssertionError("full promotion dataset이 아닙니다")
else:
    if not PILOT_INCLUDE_YOLO_CANDIDATE:
        raise RuntimeError("type-router pilot 선택에는 03C YOLO 후보가 필요합니다")
    bundle_objects = [f"dcc/mission3/runs/{BASELINE_RUN_ID}/candidate_bundle.json"]
    if INCLUDE_YOLO_CANDIDATE:
        bundle_objects.append(f"dcc/mission3/runs/{YOLO_RUN_ID}/candidate_bundle.json")
forbidden_tokens = ["test_truth", "/test_images/", "eval_private/manifests/test"]
if any(token in object_name for token in forbidden_tokens for object_name in bundle_objects):
    raise AssertionError("selection 입력에 test object가 포함됐습니다")

bundles = []
for object_name in bundle_objects:
    try:
        bundle = download_json(BUCKET, object_name)
    except NotFound as exc:
        raise FileNotFoundError(
            f"후보 bundle이 없습니다: gs://{BUCKET_NAME}/{object_name}. "
            "해당 모델 노트북의 마지막 게시 셀을 먼저 실행하십시오."
        ) from exc
    if not bundle.get("eligible_for_selection"):
        raise AssertionError(f"smoke bundle은 선택에 사용할 수 없습니다: {object_name}")
    if bundle["dataset_version"] != DATASET_VERSION:
        raise AssertionError("candidate dataset version 불일치")
    expected_scope = "full" if SELECTION_MODE == "full_promote" else "pilot"
    if bundle.get("training_scope", "pilot") != expected_scope:
        raise AssertionError("candidate training scope 불일치")
    if SELECTION_MODE == "full_promote" and bundle.get("parent_selection_id") != PILOT_SELECTION_ID:
        raise AssertionError("full candidate의 parent pilot selection 불일치")
    if bundle["validation_stem_sha256"] != dataset_seal["validation_stem_sha256"]:
        raise AssertionError("candidate와 dataset validation SHA 불일치")
    if bundle["validation_target_sha256"] != dataset_seal["validation_target_sha256"]:
        raise AssertionError("candidate와 dataset validation target SHA 불일치")
    if int(bundle.get("candidate_count", -1)) != len(bundle.get("candidates", [])):
        raise AssertionError("candidate bundle count 불일치")
    bundles.append(bundle)


## prediction 재계산과 공정 비교 hard gate

bundle에 적힌 RMSE를 그대로 신뢰하지 않고 Parquet prediction에서 다시 계산합니다.
후보마다 validation stem 집합, 행 수, type 수와 target 값이 같아야 합니다.


In [ ]:
def compute_metrics(frame):
    required = {"stem", "type", "target_m", "pred_m"}
    missing = required - set(frame.columns)
    if missing:
        raise AssertionError(f"prediction 필수 열 누락: {sorted(missing)}")
    if not frame["stem"].is_unique:
        raise AssertionError("validation stem 중복")
    if not np.isfinite(frame[["target_m", "pred_m"]].to_numpy(float)).all():
        raise AssertionError("prediction non-finite")
    error = frame["pred_m"].to_numpy(float) - frame["target_m"].to_numpy(float)
    overall = float(np.sqrt(np.mean(error ** 2)))
    by_type = {}
    for type_name in TYPE_ORDER:
        part = frame.loc[frame["type"].eq(type_name)]
        if part.empty:
            raise AssertionError(f"{type_name} validation 누락")
        e = part["pred_m"].to_numpy(float) - part["target_m"].to_numpy(float)
        by_type[type_name] = {
            "n": int(len(part)),
            "rmse_m": float(np.sqrt(np.mean(e ** 2))),
            "mae_m": float(np.mean(np.abs(e))),
            "bias_m": float(np.mean(e)),
        }
    weighted = sum(v["n"] * v["rmse_m"] ** 2 for v in by_type.values()) / len(frame)
    assert np.isclose(overall ** 2, weighted, rtol=1e-10, atol=1e-12)
    return {
        "overall_rmse_m": overall,
        "worst_type_rmse_m": max(v["rmse_m"] for v in by_type.values()),
        "by_type": by_type,
    }

candidates = []
prediction_frames = {}
reference_stems = None
reference_targets = None
reference_val_sha = None
for bundle in bundles:
    if bundle["validation_rows"] != EXPECTED_VAL_N:
        raise AssertionError("후보가 전체 validation 30,808개를 사용하지 않았습니다")
    if reference_val_sha is None:
        reference_val_sha = bundle["validation_stem_sha256"]
    elif bundle["validation_stem_sha256"] != reference_val_sha:
        raise AssertionError("candidate validation stem SHA 불일치")

    for candidate in bundle["candidates"]:
        descriptor = candidate["prediction"]
        _, object_name = parse_gs_uri(descriptor["uri"])
        local = LOCAL_INPUT / f"{candidate['model_id']}.parquet"
        download_object(BUCKET, object_name, local, generation=descriptor["generation"])
        if sha256_file(local) != descriptor["sha256"]:
            raise AssertionError("validation prediction SHA 불일치")
        frame = pd.read_parquet(local).sort_values("stem", kind="mergesort").reset_index(drop=True)
        if len(frame) != EXPECTED_VAL_N:
            raise AssertionError(f"{candidate['model_id']}: validation 행 수 오류")
        stems = frame["stem"].astype(str).tolist()
        observed_frame_sha = canonical_sha256(sorted(stems))
        if observed_frame_sha != candidate["validation_stem_sha256"]:
            raise AssertionError("prediction frame과 candidate validation SHA 불일치")
        if observed_frame_sha != dataset_seal["validation_stem_sha256"]:
            raise AssertionError("prediction frame과 dataset validation SHA 불일치")
        targets = frame[["stem", "type", "target_m"]].copy()
        observed_target_sha = canonical_target_sha256(targets, "target_m")
        if observed_target_sha != candidate["validation_target_sha256"]:
            raise AssertionError("prediction frame과 candidate validation target SHA 불일치")
        if observed_target_sha != dataset_seal["validation_target_sha256"]:
            raise AssertionError("prediction frame과 dataset validation target SHA 불일치")
        if reference_stems is None:
            reference_stems = stems
            reference_targets = targets
        else:
            if stems != reference_stems:
                raise AssertionError("candidate validation stem 순서 불일치")
            if not np.allclose(
                targets["target_m"].to_numpy(float),
                reference_targets["target_m"].to_numpy(float),
                rtol=0.0,
                atol=5e-6,
            ):
                raise AssertionError("candidate validation target 불일치")
            if targets["type"].astype(str).tolist() != reference_targets["type"].astype(str).tolist():
                raise AssertionError("candidate validation type 불일치")
        # 각 producer가 prediction Parquet의 target_m을 float32 또는 float64로
        # 저장할 수 있습니다. 먼저 해당 producer가 기록한 RMSE를 원래 target_m으로
        # 재검증한 뒤, 후보 선택과 router 합성에는 target hash 계약과 같은 float32
        # canonical truth를 사용합니다. 이렇게 해야 서로 다른 prediction 파일의
        # 미세한 target 직렬화 차이가 component RMSE assertion에 영향을 주지 않습니다.
        recorded_recomputed = compute_metrics(frame)
        recorded = candidate["metrics"]
        if not np.isclose(
            recorded_recomputed["overall_rmse_m"], recorded["overall_rmse_m"],
            rtol=1e-10, atol=1e-12,
        ):
            raise AssertionError(f"{candidate['model_id']}: 기록 RMSE 불일치")
        canonical_target_values = (
            targets["target_m"].to_numpy(dtype=np.float32).astype(np.float64)
        )
        reference_canonical_targets = (
            reference_targets["target_m"]
            .to_numpy(dtype=np.float32)
            .astype(np.float64)
        )
        if not np.array_equal(canonical_target_values, reference_canonical_targets):
            raise AssertionError("candidate canonical validation target 불일치")
        selection_frame = frame.copy()
        selection_frame["target_m"] = canonical_target_values
        selection_metrics = compute_metrics(selection_frame)
        copied = dict(candidate)
        copied["source_reported_metrics"] = recorded
        copied["source_recomputed_metrics"] = recorded_recomputed
        copied["metrics"] = selection_metrics
        copied["source_run_id"] = bundle["run_id"]
        copied["training_package_versions"] = bundle["package_versions"]
        prediction_frames[copied["model_id"]] = selection_frame
        candidates.append(copied)

raw_candidates = list(candidates)
raw_candidate_ids = [item["model_id"] for item in raw_candidates]
if len(set(raw_candidate_ids)) != len(raw_candidate_ids):
    raise AssertionError("base candidate model_id가 중복됩니다")
if (
    SELECTION_MODE == "full_promote"
    and parent_selection["winner"]["kind"] == "type_router"
):
    if len(bundles) != 2 or len(raw_candidates) != 2:
        raise AssertionError("full router는 정확히 두 component bundle/candidate가 필요합니다")
    if set(raw_candidate_ids) != set(ROUTER_COMPONENT_IDS.values()):
        raise AssertionError("full router base candidate ID 집합 오류")
    expected_source_runs = {
        ROUTER_COMPONENT_IDS["full_frame"]: BASELINE_RUN_ID,
        ROUTER_COMPONENT_IDS["yolo_roi"]: YOLO_RUN_ID,
    }
    for candidate in raw_candidates:
        if candidate["source_run_id"] != expected_source_runs[candidate["model_id"]]:
            raise AssertionError("full router component source run 불일치")

build_router = (
    SELECTION_MODE == "pilot_select"
    or parent_selection["winner"]["kind"] == "type_router"
)
router_route_by_type = None
router_candidate = None
if build_router:
    by_id = {item["model_id"]: item for item in raw_candidates}
    if not set(ROUTER_COMPONENT_IDS.values()).issubset(by_id):
        raise AssertionError(
            f"type-router component 누락: {sorted(set(ROUTER_COMPONENT_IDS.values()) - set(by_id))}"
        )
    full_component = by_id[ROUTER_COMPONENT_IDS["full_frame"]]
    yolo_component = by_id[ROUTER_COMPONENT_IDS["yolo_roi"]]
    if full_component["kind"] != "full_frame" or yolo_component["kind"] != "yolo_roi":
        raise AssertionError("type-router component kind 오류")
    full_checkpoint = full_component["checkpoint"]
    yolo_fallback = yolo_component["checkpoint_bundle"]["fallback"]
    for field in ["uri", "generation", "bytes", "sha256", "artifact_type"]:
        if yolo_fallback[field] != full_checkpoint[field]:
            raise AssertionError(f"YOLO fallback과 router full-frame checkpoint 불일치: {field}")

    if SELECTION_MODE == "pilot_select":
        router_route_by_type = {}
        for type_name in TYPE_ORDER:
            choices = [
                (
                    float(full_component["metrics"]["by_type"][type_name]["rmse_m"]),
                    0,
                    "full_frame",
                ),
                (
                    float(yolo_component["metrics"]["by_type"][type_name]["rmse_m"]),
                    1,
                    "yolo_roi",
                ),
            ]
            router_route_by_type[type_name] = min(choices)[2]
    else:
        frozen = parent_selection["winner"]
        if frozen.get("model_id") != ROUTER_MODEL_ID:
            raise AssertionError("pilot router model ID 불일치")
        router_route_by_type = dict(frozen.get("route_by_type", {}))
    if set(router_route_by_type) != set(TYPE_ORDER):
        raise AssertionError("router route type 집합 오류")
    if not set(router_route_by_type.values()).issubset({"full_frame", "yolo_roi"}):
        raise AssertionError("router route component 값 오류")

    full_frame = prediction_frames[full_component["model_id"]]
    yolo_frame = prediction_frames[yolo_component["model_id"]]
    if full_frame["stem"].tolist() != yolo_frame["stem"].tolist():
        raise AssertionError("router component stem 순서 불일치")
    router_frame = full_frame[["stem", "type", "target_m", "pred_m"]].copy()
    if not router_frame["stem"].astype(str).str[0].eq(
        router_frame["type"].astype(str)
    ).all():
        raise AssertionError("validation route type은 stem 첫 문자와 같아야 합니다")
    route_series = router_frame["type"].map(router_route_by_type)
    if route_series.isna().any():
        raise AssertionError("router route 누락")
    use_yolo = route_series.eq("yolo_roi").to_numpy()
    router_frame["pred_m"] = np.where(
        use_yolo,
        yolo_frame["pred_m"].to_numpy(float),
        full_frame["pred_m"].to_numpy(float),
    )
    router_metrics = compute_metrics(router_frame)
    router_validation_route_counts = {
        component_name: int(route_series.eq(component_name).sum())
        for component_name in ["full_frame", "yolo_roi"]
    }
    for type_name, component_name in router_route_by_type.items():
        component = full_component if component_name == "full_frame" else yolo_component
        expected_rmse = component["metrics"]["by_type"][type_name]["rmse_m"]
        observed_rmse = router_metrics["by_type"][type_name]["rmse_m"]
        if not np.isclose(observed_rmse, expected_rmse, rtol=1e-10, atol=1e-12):
            raise AssertionError(f"router {type_name} component RMSE 불일치")

    router_content_sha256 = canonical_sha256([
        {
            "stem": str(row.stem),
            "type": str(row.type),
            "target_m": float(row.target_m),
            "pred_m": float(row.pred_m),
        }
        for row in router_frame.itertuples(index=False)
    ])

    router_prediction_path = LOCAL_META / f"{ROUTER_MODEL_ID}-validation_predictions.csv"
    router_frame.to_csv(
        router_prediction_path,
        index=False,
        encoding="utf-8",
        float_format="%.17g",
        lineterminator="\n",
    )
    architecture_payload = {
        "schema_version": "dcc-m3-type-router-architecture-v1",
        "kind": "type_router",
        "routing_key": ROUTING_KEY,
        "route_by_type": router_route_by_type,
        "component_model_ids": ROUTER_COMPONENT_IDS,
        "component_architecture_fingerprints": {
            "full_frame": full_component["architecture_fingerprint"],
            "yolo_roi": yolo_component["architecture_fingerprint"],
        },
    }
    router_candidate = {
        "model_id": ROUTER_MODEL_ID,
        "kind": "type_router",
        "routing_key": ROUTING_KEY,
        "route_by_type": router_route_by_type,
        "validation_route_counts": router_validation_route_counts,
        "component_model_ids": dict(ROUTER_COMPONENT_IDS),
        "architecture_fingerprint": canonical_sha256(architecture_payload),
        "architecture_payload": architecture_payload,
        "metrics": router_metrics,
        "validation_rows": int(len(router_frame)),
        "validation_stem_sha256": dataset_seal["validation_stem_sha256"],
        "validation_target_sha256": dataset_seal["validation_target_sha256"],
        # selection hard gate 통과 후 remote descriptor로 교체합니다.
        "prediction": None,
        "prediction_format": "csv_float17g",
        "checkpoint_bundle": {
            "full_frame": full_checkpoint,
            "yolo_roi": yolo_component["checkpoint_bundle"],
        },
        "components": {
            "full_frame": full_component,
            "yolo_roi": yolo_component,
        },
        "source_run_ids": {
            "full_frame": full_component["source_run_id"],
            "yolo_roi": yolo_component["source_run_id"],
        },
        "component_training_package_versions": {
            "full_frame": full_component["training_package_versions"],
            "yolo_roi": yolo_component["training_package_versions"],
        },
        "derivation": {
            "rule": "minimum pilot validation RMSE per type; full_frame on exact tie",
            "routed_content_sha256": router_content_sha256,
            "component_prediction_sha256": {
                "full_frame": full_component["prediction"]["sha256"],
                "yolo_roi": yolo_component["prediction"]["sha256"],
            },
        },
    }
    if SELECTION_MODE == "full_promote":
        candidates = [router_candidate]
    else:
        candidates.append(router_candidate)
    print("type-router route:", json.dumps(router_route_by_type, sort_keys=True))
    print("type-router validation route counts:", router_validation_route_counts)
    print("type-router validation metrics:", json.dumps(router_metrics, indent=2))

candidate_ids = [item["model_id"] for item in candidates]
if len(set(candidate_ids)) != len(candidate_ids):
    raise AssertionError("candidate model_id가 중복됩니다")
if SELECTION_MODE == "pilot_select" and len(candidates) < 3:
    raise AssertionError("pilot 선택에는 서로 다른 배포 후보 3개 이상이 필요합니다")
if SELECTION_MODE == "full_promote":
    if len(candidates) != 1:
        raise AssertionError("full promotion에는 동결 architecture 1개만 허용합니다")
    frozen = parent_selection["winner"]
    promoted = candidates[0]
    if promoted["model_id"] != parent_selection["winner_model_id"]:
        raise AssertionError("full candidate model_id가 pilot winner와 다릅니다")
    if promoted["kind"] != frozen["kind"]:
        raise AssertionError("full candidate kind가 pilot winner와 다릅니다")
    if promoted.get("architecture_fingerprint") != frozen.get("architecture_fingerprint"):
        raise AssertionError("full candidate architecture fingerprint가 pilot과 다릅니다")
    if promoted["kind"] == "type_router":
        if promoted.get("routing_key") != frozen.get("routing_key"):
            raise AssertionError("full router routing key가 pilot과 다릅니다")
        if promoted.get("route_by_type") != frozen.get("route_by_type"):
            raise AssertionError("full router route map이 pilot과 다릅니다")
        if promoted.get("component_model_ids") != frozen.get("component_model_ids"):
            raise AssertionError("full router component model ID가 pilot과 다릅니다")
        for component_name in ["full_frame", "yolo_roi"]:
            full_component = promoted["components"][component_name]
            pilot_component = frozen["components"][component_name]
            if full_component["kind"] != pilot_component["kind"]:
                raise AssertionError(f"full router {component_name} kind 불일치")
            if full_component["architecture_fingerprint"] != pilot_component["architecture_fingerprint"]:
                raise AssertionError(f"full router {component_name} architecture 불일치")


type-router route: {"A": "yolo_roi", "B": "full_frame", "C": "yolo_roi", "E": "yolo_roi", "F": "yolo_roi", "G": "full_frame", "H": "yolo_roi", "S": "full_frame"}
type-router validation route counts: {'full_frame': 6832, 'yolo_roi': 23976}
type-router validation metrics: {
  "overall_rmse_m": 0.21439652743518847,
  "worst_type_rmse_m": 0.49531192072691393,
  "by_type": {
    "A": {
      "n": 10522,
      "rmse_m": 0.2006834520548958,
      "mae_m": 0.04633200318655051,
      "bias_m": -0.0158341751656782
    },
    "B": {
      "n": 3726,
      "rmse_m": 0.09092466149754168,
      "mae_m": 0.05894437539468705,
      "bias_m": 0.009974576648818633
    },
    "C": {
      "n": 9194,
      "rmse_m": 0.18466213136308654,
      "mae_m": 0.044785446528898834,
      "bias_m": 0.00615097024478626
    },
    "E": {
      "n": 2350,
      "rmse_m": 0.19941529545358416,
      "mae_m": 0.11189278213901722,
      "bias_m": -0.01948147305148713
    },
    "F": {
      "n": 879,
      "rmse_m": 0.107

In [ ]:
leaderboard_rows = []
for candidate in candidates:
    row = {
        "model_id": candidate["model_id"],
        "kind": candidate["kind"],
        "complexity_rank": {
            "full_frame": 1,
            "yolo_roi": 3,
            "type_router": 4,
        }[candidate["kind"]],
        "overall_rmse_m": candidate["metrics"]["overall_rmse_m"],
        "worst_type_rmse_m": candidate["metrics"]["worst_type_rmse_m"],
    }
    for type_name in TYPE_ORDER:
        row[f"rmse_{type_name}_m"] = candidate["metrics"]["by_type"][type_name]["rmse_m"]
    leaderboard_rows.append(row)
leaderboard = pd.DataFrame(leaderboard_rows).sort_values(
    ["overall_rmse_m", "worst_type_rmse_m", "complexity_rank", "model_id"],
    kind="mergesort",
).reset_index(drop=True)
display(leaderboard)
winner_id = str(leaderboard.iloc[0]["model_id"])
winner = next(item for item in candidates if item["model_id"] == winner_id)
print("validation winner:", winner_id)


                                   model_id         kind  complexity_rank  \
0  type-router-convnext-tiny-yolo26s-roi-v1  type_router                4   

   overall_rmse_m  worst_type_rmse_m  rmse_A_m  rmse_B_m  rmse_C_m  rmse_E_m  \
0        0.214397           0.495312  0.200683  0.090925  0.184662  0.199415   

   rmse_F_m  rmse_G_m  rmse_H_m  rmse_S_m  
0  0.107046  0.088632  0.495312  0.375379  

validation winner: type-router-convnext-tiny-yolo26s-roi-v1


## 선택 이유

base 후보의 pilot validation 결과를 확인한 뒤 `type_router` 구성과
유형별 경로를 정했습니다. 아래 선택 순서와 경로는 full 재학습과 test 접근
전에 pilot selection seal로 동결합니다.

1. 전체 validation meter RMSE 최소
2. 동률이면 8개 중 최악 유형 RMSE 최소
3. 다시 동률이면 구성 요소가 적고 clean runtime에서 재현된 후보

`type_router`의 유형별 경로는 pilot validation에서 ConvNeXt와 YOLO+ROI 중 해당 유형 RMSE가
낮은 component로 정하며, 정확히 같으면 full-frame을 사용합니다. 이 경로는 full 학습에서
다시 선택하지 않고 pilot seal 값을 그대로 사용합니다.

test 정답, test 파일명 구간, test annotation은 이 결정에 사용하지 않습니다.


In [ ]:
leaderboard_path = LOCAL_META / "validation_leaderboard.csv"
leaderboard.to_csv(leaderboard_path, index=False, encoding="utf-8-sig")

selection_assertions = {
    "candidate_gate_satisfied": (
        len(candidates) == 1 if SELECTION_MODE == "full_promote" else len(candidates) >= 3
    ),
    "pilot_router_candidate_present": (
        ROUTER_MODEL_ID in candidate_ids if SELECTION_MODE == "pilot_select" else True
    ),
    "winner_is_current_leaderboard_first": (
        winner_id == str(leaderboard.iloc[0]["model_id"])
    ),
    "frozen_architecture_preserved": (
        candidates[0].get("architecture_fingerprint")
        == parent_selection["winner"].get("architecture_fingerprint")
        if SELECTION_MODE == "full_promote" else True
    ),
    "validation_rows_exact": len(reference_stems) == EXPECTED_VAL_N,
    "validation_stems_identical": True,
    "validation_targets_identical": True,
    "all_predictions_finite": True,
    "router_route_types_exact": (
        set(router_route_by_type) == set(TYPE_ORDER) if build_router else True
    ),
    "router_components_pinned": (
        (
            set(raw_candidate_ids) == set(ROUTER_COMPONENT_IDS.values())
            if SELECTION_MODE == "full_promote"
            else set(ROUTER_COMPONENT_IDS.values()).issubset(set(raw_candidate_ids))
        ) if build_router else True
    ),
    "validation_route_uses_stem_first_character": (
        bool(router_frame["stem"].astype(str).str[0].eq(
            router_frame["type"].astype(str)
        ).all())
        if build_router else True
    ),
    "routing_frozen_from_pilot": (
        promoted.get("route_by_type") == parent_selection["winner"].get("route_by_type")
        if SELECTION_MODE == "full_promote" and promoted["kind"] == "type_router"
        else True
    ),
    "test_truth_not_accessed": True,
}
failed_selection_assertions = [
    key for key, value in selection_assertions.items() if not bool(value)
]
if failed_selection_assertions:
    raise AssertionError(
        f"selection hard gate 실패: {failed_selection_assertions}"
    )

# hard gate 통과 후에만 selection 파생물을 create-only로 게시합니다.
if build_router:
    router_prediction_descriptor = upload_create_only(
        BUCKET,
        router_prediction_path,
        f"{SELECTION_PREFIX}/derived/{ROUTER_MODEL_ID}/validation_predictions.csv",
        "m3_routed_validation_predictions",
    )
    router_candidate["prediction"] = router_prediction_descriptor
leaderboard_descriptor = upload_create_only(
    BUCKET,
    leaderboard_path,
    f"{SELECTION_PREFIX}/validation_leaderboard.csv",
    "m3_validation_leaderboard",
)

selection_seal = {
    "status": (
        "MODEL_DEPLOYMENT_FROZEN"
        if SELECTION_MODE == "full_promote" else "MODEL_SELECTION_FROZEN"
    ),
    "schema_version": "dcc-m3-model-selection-seal-v3",
    "selection_id": SELECTION_ID,
    "dataset_version": DATASET_VERSION,
    "training_scope": "full" if SELECTION_MODE == "full_promote" else "pilot",
    "selection_mode": SELECTION_MODE,
    "parent_selection_id": PILOT_SELECTION_ID if SELECTION_MODE == "full_promote" else None,
    "parent_selection": parent_selection_descriptor,
    "winner_model_id": winner_id,
    "winner": winner,
    "candidate_ids": candidate_ids,
    "candidate_count": len(candidates),
    "base_candidate_ids": raw_candidate_ids,
    "base_candidate_count": len(raw_candidates),
    "validation_rows": EXPECTED_VAL_N,
    "validation_stem_sha256": reference_val_sha,
    "validation_target_sha256": dataset_seal["validation_target_sha256"],
    "selection_rule": [
        "minimum validation overall meter RMSE",
        "minimum worst-type meter RMSE",
        "simpler reproducible pipeline if still tied",
        "type-router route: minimum pilot RMSE per type, full-frame on exact tie",
    ],
    "candidate_metrics": {
        item["model_id"]: item["metrics"] for item in candidates
    },
    "candidate_bundle_objects": bundle_objects,
    "base_candidate_metrics": {
        item["model_id"]: item["metrics"] for item in raw_candidates
    },
    "leaderboard": leaderboard_descriptor,
    "test_inputs_accessed": False,
    "assertions": selection_assertions,
}
selection_descriptor = upload_json_create_only(
    BUCKET,
    selection_seal,
    f"{SELECTION_PREFIX}/model_selection_seal.json",
    "m3_model_selection_seal",
)
print(selection_seal["status"])
print("winner:", winner_id)
print("selection seal:", selection_descriptor["uri"])


MODEL_DEPLOYMENT_FROZEN
winner: type-router-convnext-tiny-yolo26s-roi-v1
selection seal: gs://YOUR_GCS_BUCKET/dcc/mission3/selection/m3-selection-full-router-v2/model_selection_seal.json


## 다음 단계

`pilot_select`가 끝나면 03A를 `full`로 실행합니다. type-router 승자는 03B의 ConvNeXt
`full_retrain`을 먼저 끝내고, 그 bundle을 fallback으로 읽는 03C의 YOLO+ROI `full_retrain`을
이어서 실행합니다. 그 뒤 이 노트북을 `full_promote`로 다시 실행합니다.
`MODEL_DEPLOYMENT_FROZEN`이 출력된 다음에만 03E를 새 GPU runtime에서 실행합니다.
